In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn import tree
#import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten,Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle
from sklearn.model_selection import GridSearchCV




## Load data

In [2]:
train_data=np.load('Data+Description\\fashion_train.npy')  
train_data_X=train_data[:,:-1]
train_data_y=train_data[:,-1]

In [3]:
with open("pca_model_50_components.pkl", "rb") as file:
    pca_loaded = pickle.load(file)

# Apply the loaded PCA model to new data
X_new_reduced = pca_loaded.transform(train_data_X)  # X_new is new data


In [19]:
var_cumu = np.cumsum(pca_loaded.explained_variance_ratio_)*100
print(var_cumu)
k = np.argmax(var_cumu>99)
print("Number of components explaining 95% variance: "+ str(k))

[32.30994384 48.57960044 56.31012675 60.47880909 63.76410878 66.00166641
 67.94284694 69.72410797 71.22079556 72.49360387 73.63000821 74.5998405
 75.4322463  76.17536213 76.90677937 77.58403945 78.25323061 78.8900573
 79.45595373 79.95952339 80.44330693 80.90572714 81.35286908 81.77975969
 82.19907905 82.59517338 82.96879551 83.31153025 83.64223557 83.949844
 84.24883573 84.53881637 84.81803706 85.09251581 85.36176285 85.61501275
 85.86077114 86.09212551 86.31923504 86.53695104 86.74258443 86.94614116
 87.14544106 87.33923074 87.53106054 87.71979325 87.89738887 88.06795005
 88.23295724 88.39017932]
Number of components explaining 95% variance: 0


In [7]:
print(X_new_reduced.shape)
print(train_data_X.shape)


(10000, 50)
(10000, 784)


## Tree

In [6]:
tree_classifier=tree.DecisionTreeClassifier()
scores = cross_validate(tree_classifier, train_data_X, train_data_y, cv=5)

In [ ]:
print(scores)
print("test_score_avg - ", sum(scores['test_score'])/5)

{'fit_time': array([ 9.92942882, 10.07927227, 10.60560369, 11.47036314, 10.76992965]),
 'score_time': array([0.01673245, 0.01100111, 0.00800061, 0.00798082, 0.00700355]),
 'test_score': array([0.7775, 0.7815, 0.7805, 0.77  , 0.7715])}

In [6]:
print(scores)
print("test_score_avg - ", sum(scores['test_score'])/5)

{'fit_time': array([10.41250777, 10.65960598, 10.82245636, 11.84291673, 10.91298938]), 'score_time': array([0.01871014, 0.00700068, 0.0080061 , 0.00700402, 0.00799608]), 'test_score': array([0.7765, 0.789 , 0.783 , 0.777 , 0.7945])}
test_score_avg -  0.784


In [ ]:
tree_classifier=tree.DecisionTreeClassifier()
scores_pca = cross_validate(tree_classifier, X_new_reduced, train_data_y, cv=5)
print(scores_pca)
print("test_score_avg - ", sum(scores['test_score'])/5)

{'fit_time': array([2.19031692, 2.17263365, 2.08894372, 2.1109817 , 2.01655817]), 'score_time': array([0.01265144, 0.00200391, 0.00200129, 0.0020113 , 0.00200462]), 'test_score': array([0.747 , 0.763 , 0.744 , 0.7665, 0.75  ])}
test_score_avg -  0.7541


## Grid search for trees

In [10]:
param_grid = {
    'criterion': ['gini','entropy'],
    'max_depth': [ 5, 10, 15, 25, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# Initialize the Decision Tree model
dt = tree.DecisionTreeClassifier()

# Perform grid search with 5-fold cross-validation
grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_new_reduced, train_data_y)

# Best parameters and score
print("Best parameters:", grid_search.best_params_)
print("Best cross-validated accuracy score:", grid_search.best_score_)

Best parameters: {'criterion': 'entropy', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 5}
Best cross-validated accuracy score: 0.7695000000000001


## FFN

In [ ]:
"""def create_model(input_dim):
    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))  # Output layer for binary classification
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model"""

def create_multiclass_model(input_dim, num_classes):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))  # First hidden layer
    model.add(Dense(64, activation='relu'))  # Second hidden layer
    model.add(Dense(32, activation='relu'))  # Third hidden layer
    model.add(Dense(num_classes, activation='softmax'))  # Output layer for multi-class classification
    
    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',  # or 'categorical_crossentropy' if using one-hot encoding
                  metrics=['accuracy'])
    return model

In [8]:

# Set up 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = train_data_X[train_index], train_data_X[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=train_data_X.shape[1], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Get the predicted class for each sample

    # Calculate accuracy for the current fold
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

Training fold 1


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 1 Accuracy: 0.8390
Classification Report for Fold 1:
               precision    recall  f1-score   support

           0       0.84      0.82      0.83       415
           1       1.00      0.96      0.98       379
           2       0.77      0.86      0.82       388
           3       0.90      0.89      0.90       405
           4       0.70      0.68      0.69       413

    accuracy                           0.84      2000
   macro avg       0.84      0.84      0.84      2000
weighted avg       0.84      0.84      0.84      2000

Confusion Matrix for Fold 1:
 [[339   0  18   7  51]
 [  0 364   5   6   4]
 [  2   0 335   4  47]
 [  8   1  18 360  18]
 [ 55   0  57  21 280]]
Training fold 2


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 2 Accuracy: 0.8195
Classification Report for Fold 2:
               precision    recall  f1-score   support

           0       0.73      0.79      0.76       398
           1       0.99      0.97      0.98       393
           2       0.77      0.92      0.84       425
           3       0.88      0.86      0.87       396
           4       0.73      0.54      0.62       388

    accuracy                           0.82      2000
   macro avg       0.82      0.82      0.81      2000
weighted avg       0.82      0.82      0.82      2000

Confusion Matrix for Fold 2:
 [[316   0  13  26  43]
 [  2 382   3   5   1]
 [  6   0 392   4  23]
 [ 32   0  13 339  12]
 [ 77   2  87  12 210]]
Training fold 3


...

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Fold 3 Accuracy: 0.8200
Classification Report for Fold 3:
               precision    recall  f1-score   support

           0       0.75      0.84      0.79       399
           1       0.97      0.99      0.98       398
           2       0.73      0.89      0.80       366
           3       0.89      0.88      0.88       398
           4       0.77      0.54      0.63       439

    accuracy                           0.82      2000
   macro avg       0.82      0.83      0.82      2000
weighted avg       0.82      0.82      0.81      2000

Confusion Matrix for Fold 3:
 [[335   1  14  17  32]
 [  1 393   1   2   1]
 [  6   1 327   5  27]
 [ 16  10  14 349   9]
 [ 88   2  94  19 236]]
Training fold 4


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 4 Accuracy: 0.8550
Classification Report for Fold 4:
               precision    recall  f1-score   support

           0       0.81      0.87      0.84       411
           1       0.98      0.93      0.96       414
           2       0.86      0.87      0.87       406
           3       0.88      0.90      0.89       396
           4       0.73      0.69      0.71       373

    accuracy                           0.85      2000
   macro avg       0.85      0.85      0.85      2000
weighted avg       0.86      0.85      0.85      2000

Confusion Matrix for Fold 4:
 [[357   1   4  11  38]
 [  6 386   0  21   1]
 [  7   0 353   4  42]
 [ 16   5   3 357  15]
 [ 54   0  50  12 257]]
Training fold 5


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 5 Accuracy: 0.8290
Classification Report for Fold 5:
               precision    recall  f1-score   support

           0       0.83      0.81      0.82       410
           1       0.99      0.95      0.97       363
           2       0.87      0.78      0.82       416
           3       0.88      0.91      0.90       410
           4       0.62      0.70      0.66       401

    accuracy                           0.83      2000
   macro avg       0.84      0.83      0.83      2000
weighted avg       0.84      0.83      0.83      2000

Confusion Matrix for Fold 5:
 [[334   0   0  17  59]
 [  0 346   0  13   4]
 [  6   0 324   4  82]
 [  6   2   0 374  28]
 [ 57   0  47  17 280]]

Average Accuracy across 5 folds: 0.8325


In [10]:
# pca
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(X_new_reduced)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = X_new_reduced[train_index], X_new_reduced[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=X_new_reduced.shape[1], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Get the predicted class for each sample

    # Calculate accuracy for the current fold
    accuracy_pca = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy_pca)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy_pca = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

Training fold 1


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 1 Accuracy: 0.8290
Classification Report for Fold 1:
               precision    recall  f1-score   support

           0       0.78      0.79      0.78       415
           1       0.95      0.96      0.95       379
           2       0.81      0.82      0.82       388
           3       0.87      0.86      0.87       405
           4       0.67      0.65      0.66       413

    accuracy                           0.81      2000
   macro avg       0.82      0.82      0.82      2000
weighted avg       0.81      0.81      0.81      2000

Confusion Matrix for Fold 1:
 [[327   4  12  21  51]
 [  5 362   4   5   3]
 [  4   5 320   6  53]
 [ 17   5   9 350  24]
 [ 68   4  50  21 270]]
Training fold 2


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 2 Accuracy: 0.8290
Classification Report for Fold 2:
               precision    recall  f1-score   support

           0       0.80      0.65      0.72       398
           1       0.97      0.98      0.98       393
           2       0.77      0.88      0.82       425
           3       0.84      0.83      0.84       396
           4       0.62      0.65      0.64       388

    accuracy                           0.80      2000
   macro avg       0.80      0.80      0.80      2000
weighted avg       0.80      0.80      0.80      2000

Confusion Matrix for Fold 2:
 [[258   0  18  27  95]
 [  1 386   0   6   0]
 [  4   2 375  11  33]
 [ 23   7  13 327  26]
 [ 37   2  80  16 253]]
Training fold 3


...

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 3 Accuracy: 0.8290
Classification Report for Fold 3:
               precision    recall  f1-score   support

           0       0.77      0.77      0.77       399
           1       0.97      0.96      0.97       398
           2       0.78      0.82      0.80       366
           3       0.87      0.88      0.88       398
           4       0.68      0.65      0.66       439

    accuracy                           0.81      2000
   macro avg       0.81      0.82      0.82      2000
weighted avg       0.81      0.81      0.81      2000

Confusion Matrix for Fold 3:
 [[308   1   9  16  65]
 [  1 381   2  12   2]
 [  4   3 300   5  54]
 [ 18   5  10 350  15]
 [ 71   1  62  19 286]]
Training fold 4


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 4 Accuracy: 0.8290
Classification Report for Fold 4:
               precision    recall  f1-score   support

           0       0.77      0.80      0.79       411
           1       0.97      0.96      0.96       414
           2       0.86      0.78      0.82       406
           3       0.84      0.89      0.86       396
           4       0.66      0.67      0.66       373

    accuracy                           0.82      2000
   macro avg       0.82      0.82      0.82      2000
weighted avg       0.82      0.82      0.82      2000

Confusion Matrix for Fold 4:
 [[330   0   7  25  49]
 [  5 396   0  12   1]
 [ 12   1 316   7  70]
 [ 21   7   6 351  11]
 [ 59   4  38  22 250]]
Training fold 5


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 5 Accuracy: 0.8290
Classification Report for Fold 5:
               precision    recall  f1-score   support

           0       0.81      0.69      0.74       410
           1       0.95      0.93      0.94       363
           2       0.84      0.75      0.80       416
           3       0.82      0.89      0.85       410
           4       0.60      0.72      0.66       401

    accuracy                           0.79      2000
   macro avg       0.80      0.80      0.80      2000
weighted avg       0.80      0.79      0.80      2000

Confusion Matrix for Fold 5:
 [[281   5   5  26  93]
 [  1 339   4  12   7]
 [ 13   2 314  17  70]
 [  7  10   9 363  21]
 [ 45   2  41  23 290]]

Average Accuracy across 5 folds: 0.8325


In [ ]:
fnn_classifier = create_multiclass_model()

In [ ]:
scores

## CNN

In [4]:

def create_cnn_model(input_shape, num_classes):
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(256, activation='relu'))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  # Softmax for multi-class classification
    
    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001), 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = train_data_X[train_index].reshape(-1, 28, 28,1), train_data_X[val_index].reshape(-1, 28, 28,1)
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    #print(X_train[0])
    
    # Create a new instance of the CNN model
    model = create_cnn_model(input_shape=train_data_X.reshape(-1, 28, 28, 1).shape[1:], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Convert probabilities to class predictions
    
    # Calculate accuracy for the current fold
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

Training fold 1


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 1 Accuracy: 0.8620
Classification Report for Fold 1:
               precision    recall  f1-score   support

           0       0.77      0.90      0.83       415
           1       0.98      0.96      0.97       379
           2       0.89      0.88      0.89       388
           3       0.91      0.88      0.90       405
           4       0.78      0.69      0.73       413

    accuracy                           0.86      2000
   macro avg       0.87      0.86      0.86      2000
weighted avg       0.86      0.86      0.86      2000

Confusion Matrix for Fold 1:
 [[374   0   4   7  30]
 [  4 365   1   7   2]
 [  8   0 340   7  33]
 [ 20   7   4 358  16]
 [ 81   0  31  14 287]]
Training fold 2


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 2 Accuracy: 0.8475
Classification Report for Fold 2:
               precision    recall  f1-score   support

           0       0.80      0.74      0.77       398
           1       0.98      0.99      0.98       393
           2       0.94      0.82      0.87       425
           3       0.92      0.91      0.92       396
           4       0.64      0.78      0.70       388

    accuracy                           0.85      2000
   macro avg       0.86      0.85      0.85      2000
weighted avg       0.86      0.85      0.85      2000

Confusion Matrix for Fold 2:
 [[296   1   5   8  88]
 [  1 388   0   4   0]
 [  6   0 347   3  69]
 [ 16   3   1 361  15]
 [ 49   3  16  17 303]]
Training fold 3


...

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 3 Accuracy: 0.8430
Classification Report for Fold 3:
               precision    recall  f1-score   support

           0       0.87      0.73      0.79       399
           1       0.98      0.97      0.98       398
           2       0.80      0.90      0.85       366
           3       0.84      0.95      0.89       398
           4       0.74      0.68      0.71       439

    accuracy                           0.84      2000
   macro avg       0.84      0.85      0.84      2000
weighted avg       0.84      0.84      0.84      2000

Confusion Matrix for Fold 3:
 [[291   3  16  22  67]
 [  1 387   0   8   2]
 [  4   0 330   7  25]
 [  2   2   6 379   9]
 [ 38   2  63  37 299]]
Training fold 4


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 4 Accuracy: 0.8750
Classification Report for Fold 4:
               precision    recall  f1-score   support

           0       0.84      0.87      0.85       411
           1       0.98      0.97      0.97       414
           2       0.87      0.87      0.87       406
           3       0.93      0.92      0.92       396
           4       0.74      0.74      0.74       373

    accuracy                           0.88      2000
   macro avg       0.87      0.87      0.87      2000
weighted avg       0.88      0.88      0.88      2000

Confusion Matrix for Fold 4:
 [[356   3  11   9  32]
 [  3 402   1   5   3]
 [  4   0 354   2  46]
 [  9   5   5 363  14]
 [ 52   1  35  10 275]]
Training fold 5


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 5 Accuracy: 0.8490
Classification Report for Fold 5:
               precision    recall  f1-score   support

           0       0.86      0.83      0.84       410
           1       1.00      0.95      0.97       363
           2       0.93      0.72      0.82       416
           3       0.89      0.92      0.91       410
           4       0.65      0.83      0.73       401

    accuracy                           0.85      2000
   macro avg       0.87      0.85      0.85      2000
weighted avg       0.86      0.85      0.85      2000

Confusion Matrix for Fold 5:
 [[340   0   2   8  60]
 [  1 345   0  12   5]
 [ 21   0 301   8  86]
 [  5   0   1 379  25]
 [ 30   0  18  20 333]]

Average Accuracy across 5 folds: 0.8553


In [24]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

#ipca = IncrementalPCA(n_components=50)
#X_new_reduced_CNN=(X_new_reduced).reshape(-1, 5, 10, 1)

for fold, (train_index, val_index) in enumerate(kf.split(X_new_reduced)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = X_new_reduced[train_index].reshape(-1, 5, 10, 1), X_new_reduced[val_index].reshape(-1, 5, 10, 1)
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    #print(X_train[0])
    
    # Create a new instance of the CNN model
    model = create_cnn_model(input_shape=X_new_reduced.reshape(-1, 5, 10, 1).shape[1:], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Convert probabilities to class predictions
    
    # Calculate accuracy for the current fold
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

Training fold 1


ValueError: Computed output size would be negative. Received `inputs shape=(None, 1, 4, 32)`, `kernel shape=(3, 3, 32, 64)`, `dilation_rate=[1 1]`.